## **1-1. CSV 파일 읽기**

### ① 파일 읽기에 실패한 경우

In [1]:
import pandas as pd

try:
    # ①, ②: 파일 읽기 및 df_house에 저장
    df_house = pd.read_csv('1.01.house.price_error.csv')

    # ③: 첫 번째 행과 전체 데이터 행 수 출력
    print("▶ 첫 번째 행 출력")
    display(df_house.head(1))

    print("\n▶ 전체 데이터 행 수")
    print(len(df_house))

    # ④: 첫 번째 행의 price 값 자료형 출력
    print("\n▶ 첫 번째 행의 price 값 자료형")
    print(type(df_house.loc[0, 'price']))

except FileNotFoundError:
    # ⑤: 파일이 존재하지 않을 때 실행
    print("파일명과 업로드 여부를 확인하세요.")

파일명과 업로드 여부를 확인하세요.


### ② 파일 읽기에 성공한 경우

In [2]:
import pandas as pd

try:
    # ①, ②: 파일 읽기 및 df_house에 저장
    df_house = pd.read_csv('1.01.house.price.csv')

    # ③: 첫 번째 행과 전체 데이터 행 수 출력
    print("▶ 첫 번째 행 출력")
    display(df_house.head(1))

    print("\n▶ 전체 데이터 행 수")
    print(len(df_house))

    # ④: 첫 번째 행의 price 값 자료형 출력
    print("\n▶ 첫 번째 행의 price 값 자료형")
    print(type(df_house.loc[0, 'price']))

except FileNotFoundError:
    # ⑤: 파일이 존재하지 않을 때 실행
    print("파일명과 업로드 여부를 확인하세요.")

▶ 첫 번째 행 출력


,ID,size,year,price
0,1,34,5,5.0



▶ 전체 데이터 행 수
10

▶ 첫 번째 행의 price 값 자료형
<class 'numpy.float64'>


## **1-2. 문자열을 숫자로 변환하기**

In [3]:
# ① 빈 리스트 houses 생성
houses = []

# 정상 저장 행 수와 제외 행 수 카운트 변수
valid_count = 0
invalid_count = 0

# ② 반복문을 사용하여 각 행을 순회
for idx, row in df_house.iterrows():
    try:
        # ④ 필수 값 누락 확인
        if row[['ID', 'size', 'year', 'price']].isna().any():
            raise ValueError("필수 값 누락")

        # ③ 각 열의 자료형 변환
        house_item = {
            'ID': int(row['ID']),
            'size': int(row['size']),
            'year': int(row['year']),
            'price': float(row['price'])
        }

        # 정상 데이터 houses 리스트에 추가
        houses.append(house_item)
        valid_count += 1

    # ④, ⑤ 변환 오류 시 제외 사유 출력 및 계속 진행
    except Exception as e:
        invalid_count += 1
        print(f"[제외] 행 번호(인덱스): {idx} / 사유: {e}")

# ⑤ 정상 저장 행 수와 제외 행 수 출력
print("="*30)
print(f"정상 저장 행 수: {valid_count}")
print(f"제외 행 수: {invalid_count}")
print("="*30)

# 변환된 첫 번째 행의 각 열 자료형 확인
if houses:
    print("\n▶ 변환된 첫 번째 행 자료형 확인")
    first_item = houses[0]
    for key, value in first_item.items():
        print(f"{key}: {value} ({type(value).__name__})")

정상 저장 행 수: 10
제외 행 수: 0

▶ 변환된 첫 번째 행 자료형 확인
ID: 1 (int)
size: 34 (int)
year: 5 (int)
price: 5.0 (float)


## **1-3. 조건에 맞는 주택 찾기**

In [4]:
import copy

# ① 원본 houses 유지 및 테스트 데이터 추가
test_data = [
    {"ID": 101, "year": 5, "price": 3},
    {"ID": 102, "size": 35, "year": 5, "price": None},
    {"ID": 103, "size": "오류", "year": 5, "price": 3},
    {"ID": 104, "size": 35, "year": 5, "price": 3}
]

houses_test = copy.deepcopy(houses)
houses_test.extend(test_data)

selected = []

print("▶ 오류 데이터 제외 사유")
for house in houses_test:
    house_id = house.get('ID', '알 수 없음')

    # 1. 키 누락 검사 (or 사용)
    if ('size' not in house) or ('price' not in house):
        print(f"[제외] ID {house_id}: size 또는 price 키 누락")
        continue

    # 2. None / 결측값 검사 (or 사용)
    if (house['size'] is None) or (house['price'] is None):
        print(f"[제외] ID {house_id}: size 또는 price 값이 None")
        continue

    # 3. 숫자로 변환 가능한지 검사
    try:
        size_val = float(house['size'])
        price_val = float(house['price'])
    except (ValueError, TypeError):
        print(f"[제외] ID {house_id}: 숫자로 변환할 수 없는 값")
        continue

    # 4. size가 0 이하이거나 price가 음수인 경우 (or 사용)
    if size_val <= 0 or price_val < 0:
        print(f"[제외] ID {house_id}: size가 0 이하이거나 price가 음수")
        continue

    # 5. size가 30 이상이고 price가 4 이하 (and 사용)
    if size_val >= 30 and price_val <= 4:
        selected.append(house)

# ⑥ 선택한 주택의 ID 출력
print("\n▶ 선택된 주택 ID")
for h in selected:
    print(f"ID: {h['ID']}")

# ⑦ 전체 건수 출력
print(f"\n▶선택된 주택의 전체 건수\n{len(selected)}건")

▶ 오류 데이터 제외 사유
[제외] ID 101: size 또는 price 키 누락
[제외] ID 102: size 또는 price 값이 None
[제외] ID 103: 숫자로 변환할 수 없는 값

▶ 선택된 주택 ID
ID: 3
ID: 4
ID: 5
ID: 9
ID: 10
ID: 104

▶선택된 주택의 전체 건수
6건


## **1-4. 함수와 예외 처리**

### ① 전체 houses의 크기당 가격

In [8]:
# ①~⑦ unit_price 함수 작성
def unit_price(price, size):
    # ②, ⑤, ⑥ 숫자 변환 및 자료형/값 오류 예외 처리
    try:
        p = float(price)
        s = float(size)
    except ValueError:
        print("값 오류: 숫자로 변환할 수 없는 문자열입니다.")
        return None
    except TypeError:
        print("자료형 오류: 숫자 또는 숫자 형태의 문자열을 입력하세요.")
        return None

    # ④ 크기 및 가격 유효성 검사
    if s <= 0:
        print("크기 오류: 크기가 0 이하입니다.")
        return None
    if p < 0:
        print("가격 오류: 가격이 음수입니다.")
        return None

    # ③ 정상 입력 시 크기당 가격(가격 ÷ 크기) 반환
    return p / s

    # ⑧ 전체 houses에 적용하여 주택 ID와 크기당 가격 출력
print("▶ 전체 houses의 크기당 가격")
for h in houses:
    u_price = unit_price(h['price'], h['size'])
    if u_price is not None:
        print(f"ID {h['ID']}: 크기당 가격 = {u_price:.4f}")

▶ 전체 houses의 크기당 가격
ID 1: 크기당 가격 = 0.1471
ID 2: 크기당 가격 = 0.1000
ID 3: 크기당 가격 = 0.1333
ID 4: 크기당 가격 = 0.0789
ID 5: 크기당 가격 = 0.0750
ID 6: 크기당 가격 = 0.0875
ID 7: 크기당 가격 = 0.0885
ID 8: 크기당 가격 = 0.1000
ID 9: 크기당 가격 = 0.0882
ID 10: 크기당 가격 = 0.0853


### ② 함수 호출 및 실행 결과

In [18]:
print("1.")
print(unit_price(4, 40))
print("\n2.")
print(unit_price("4", "40"))
print("\n3.")
print(unit_price(4, 0))
print("\n4.")
print(unit_price(-4, 40))
print("\n5.")
print(unit_price("오류", 40))
print("\n6.")
print(unit_price(None, 40))
print("\n7.")
print(unit_price(6, 30))

1.
0.1

2.
0.1

3.
크기 오류: 크기가 0 이하입니다.
None

4.
가격 오류: 가격이 음수입니다.
None

5.
값 오류: 숫자로 변환할 수 없는 문자열입니다.
None

6.
자료형 오류: 숫자 또는 숫자 형태의 문자열을 입력하세요.
None

7.
0.2


## **1-5. 목적에 맞는 자료구조 선택하기**

In [20]:
# ==============================================================================
# ① 전체 주택 ID를 원래 순서대로 저장하고 출력
# 선택 자료구조: 리스트(list)
# 선택 이유: 원소의 삽입 순서(원래 순서)가 보장되고 인덱스를 통한 순차적 접근이 가능하기 때문.
# 대안과의 차이: 집합(set)은 중복을 제거하지만 순서를 보장하지 않으므로 본 목적에 부적합함.

house_ids = [house['ID'] for house in houses]
print("▶ 전체 주택 ID 목록 (리스트)")
print(house_ids)
# ==============================================================================


# ==============================================================================
# ② 주택의 year 값을 중복 없이 저장하고 출력
# 선택 자료구조: 집합(set)
# 선택 이유: 중복을 허용하지 않고 고유한 값만을 자동으로 필터링하여 유지하기 때문.
# 대안과의 차이: 리스트(list)는 중복 원소를 그대로 허용하므로 중복 제거를 위해 추가 작업이 필요함.

unique_years = {house['year'] for house in houses}
print("\n▶ 중복 없는 year 값 (집합)")
print(unique_years)
# ==============================================================================


# ==============================================================================
# ③ 주택 ID로 가격을 조회할 수 있도록 저장하고, ID 3의 가격 출력
# 선택 자료구조: 딕셔너리(dictionary)
# 선택 이유: Key(주택 ID)-Value(가격) 매핑을 통해 특정 ID의 가격을 O(1)의 빠른 속도로 직접 조회할 수 있기 때문.
# 대안과의 차이: 튜플(tuple)이나 리스트는 특정 ID 값을 찾으려면 전체 데이터를 순회(O(N))해야 하므로 키 기반 검색에 비효율적임.

id_to_price = {house['ID']: house['price'] for house in houses}
print("\n▶ ID 3의 가격 (딕셔너리 조회)")
print(id_to_price[3])
# ==============================================================================


# ==============================================================================
# ④ 최소 크기(30)와 최대 가격(4)을 원소로 갖는 불변 묶음 저장
# 선택 자료구조: 튜플(tuple)
# 선택 이유: 불변(immutable) 자료형으로, 한 번 생성된 후 원소 추가/삭제/교체 등 임의 수정을 원천 차단하기 때문.
# 대안과의 차이: 리스트(list)는 가변(mutable) 자료형이라 append(), pop() 등을 통해 값이 쉽게 변형될 위험이 있음.

search_criteria = (30, 4)
print("\n▶ 불변 검색 조건 (튜플)")
print(search_criteria)
# ==============================================================================

▶ 전체 주택 ID 목록 (리스트)
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

▶ 중복 없는 year 값 (집합)
{2, 5, 12, 18, 19, 20, 22}

▶ ID 3의 가격 (딕셔너리 조회)
4.0

▶ 불변 검색 조건 (튜플)
(30, 4)
